# 🎯 F1 Feature Importance Analysis

**Goal:** Determine which features are most predictive of qualifying/race performance

**Methods:**
1. Univariate feature selection (statistical tests)
2. Tree-based feature importance (Random Forest)
3. Permutation importance (model-agnostic)
4. SHAP values (interpretable ML)
5. Feature interaction analysis
6. Consolidated ranking

In [ ]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_regression, f_regression
import shap

import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")

---
## 1️⃣ Load and Prepare Data

In [ ]:
# Load features
df = pd.read_parquet('../data/features/ml_features_2022_2025.parquet')

# Focus on qualifying for this analysis
df_qual = df[df['qualifying_position'].notna()].copy()

print(f"📦 Qualifying dataset: {df_qual.shape}")
print(f"📅 Years: {df_qual['year'].min()} - {df_qual['year'].max()}")

# Define target
target_col = 'qualifying_position'

# Define feature groups
historical_features = [
    'circuit_avg_position', 'circuit_best_position', 'circuit_position_std',
    'recent_avg_position', 'recent_best_position', 'form_trend',
    'wet_dry_delta', 'team_circuit_avg_position', 'team_momentum'
]

telemetry_features = [
    'max_throttle_ratio', 'braking_events', 'brake_max_g', 'brake_avg_g',
    'drs_activations', 'degradation_slope'
]

weather_features = [
    'avg_rainfall', 'avg_track_temp', 'avg_air_temp'
]

tire_features = [
    'tyre_age', 'is_fresh_tyre'
]

# Filter to existing columns
historical_features = [f for f in historical_features if f in df_qual.columns]
telemetry_features = [f for f in telemetry_features if f in df_qual.columns]
weather_features = [f for f in weather_features if f in df_qual.columns]
tire_features = [f for f in tire_features if f in df_qual.columns]

# All numeric features
all_features = historical_features + telemetry_features + weather_features + tire_features

print(f"\n📊 Feature counts:")
print(f"   Historical: {len(historical_features)}")
print(f"   Telemetry: {len(telemetry_features)}")
print(f"   Weather: {len(weather_features)}")
print(f"   Tire: {len(tire_features)}")
print(f"   Total: {len(all_features)}")

In [ ]:
# Prepare modeling dataset
# Drop rows with missing target
df_model = df_qual[all_features + [target_col]].copy()

# Fill missing values with median (simple imputation)
for col in all_features:
    if df_model[col].isnull().any():
        df_model[col].fillna(df_model[col].median(), inplace=True)

print(f"\n📦 Model dataset: {df_model.shape}")
print(f"✅ Missing values: {df_model.isnull().sum().sum()}")

---
## 2️⃣ Univariate Feature Selection

In [ ]:
# F-statistic (linear correlation)
X = df_model[all_features]
y = df_model[target_col]

f_scores, p_values = f_regression(X, y)

f_importance = pd.DataFrame({
    'feature': all_features,
    'f_score': f_scores,
    'p_value': p_values
}).sort_values('f_score', ascending=False)

# Plot
fig = px.bar(
    f_importance.head(15),
    x='f_score',
    y='feature',
    orientation='h',
    title='📊 F-Statistic: Linear Relationship with Target',
    labels={'f_score': 'F-Score', 'feature': 'Feature'},
    color='f_score',
    color_continuous_scale='Viridis',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n🔝 Top 10 features by F-statistic:")
print(f_importance.head(10)[['feature', 'f_score', 'p_value']].to_string(index=False))

In [ ]:
# Mutual Information (non-linear relationships)
mi_scores = mutual_info_regression(X, y, random_state=42)

mi_importance = pd.DataFrame({
    'feature': all_features,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

# Plot
fig = px.bar(
    mi_importance.head(15),
    x='mi_score',
    y='feature',
    orientation='h',
    title='🔗 Mutual Information: Non-Linear Relationships',
    labels={'mi_score': 'MI Score', 'feature': 'Feature'},
    color='mi_score',
    color_continuous_scale='Plasma',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n🔝 Top 10 features by Mutual Information:")
print(mi_importance.head(10).to_string(index=False))

---
## 3️⃣ Tree-Based Feature Importance

In [ ]:
# Train Random Forest
print("🌲 Training Random Forest...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

train_score = rf.score(X_train, y_train)
test_score = rf.score(X_test, y_test)

print(f"✅ Train R²: {train_score:.3f}")
print(f"✅ Test R²: {test_score:.3f}")

In [ ]:
# Feature importance from Random Forest
rf_importance = pd.DataFrame({
    'feature': all_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# Plot
fig = px.bar(
    rf_importance.head(15),
    x='importance',
    y='feature',
    orientation='h',
    title='🌲 Random Forest Feature Importance',
    labels={'importance': 'Importance', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='Greens',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n🔝 Top 10 features by Random Forest:")
print(rf_importance.head(10).to_string(index=False))

---
## 4️⃣ Permutation Importance

In [ ]:
# Permutation importance (model-agnostic)
print("🔀 Computing permutation importance...")

perm_importance = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': all_features,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("✅ Done!")

In [ ]:
# Plot permutation importance with error bars
top_perm = perm_df.head(15)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_perm['importance_mean'],
    y=top_perm['feature'],
    orientation='h',
    error_x=dict(type='data', array=top_perm['importance_std']),
    marker_color='indianred'
))

fig.update_layout(
    title='🔀 Permutation Importance (with std dev)',
    xaxis_title='Importance',
    yaxis_title='Feature',
    height=600,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

print("\n🔝 Top 10 features by Permutation Importance:")
print(perm_df.head(10)[['feature', 'importance_mean', 'importance_std']].to_string(index=False))

---
## 5️⃣ SHAP Values (Interpretable ML)

In [ ]:
# Compute SHAP values (sample subset for speed)
print("🎯 Computing SHAP values...")

# Sample 1000 instances for SHAP (full dataset is slow)
X_shap = X_test.sample(min(1000, len(X_test)), random_state=42)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)

print("✅ Done!")

In [ ]:
# SHAP feature importance (mean absolute SHAP values)
shap_importance = pd.DataFrame({
    'feature': all_features,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

# Plot with Plotly
fig = px.bar(
    shap_importance.head(15),
    x='shap_importance',
    y='feature',
    orientation='h',
    title='🎯 SHAP Feature Importance (Mean |SHAP value|)',
    labels={'shap_importance': 'Mean |SHAP Value|', 'feature': 'Feature'},
    color='shap_importance',
    color_continuous_scale='Reds',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print("\n🔝 Top 10 features by SHAP:")
print(shap_importance.head(10).to_string(index=False))

In [ ]:
# SHAP beeswarm plot 
# For each feature, show distribution of SHAP values colored by feature value

import numpy as np
import plotly.express as px
import pandas as pd

# ... your code above that builds plot_df ...

# --- NEW: percentile per feature, then bin into 9 buckets (discrete colors) ---
plot_df['feature_value_pct'] = (
    plot_df.groupby('feature')['feature_value']
           .transform(lambda s: s.rank(pct=True, method='average'))
)

bins = [0, .125, .25, .375, .5, .625, .75, .875, 1.0]
labels = [f'{int(bins[i]*100)}–{int(bins[i+1]*100)}%' for i in range(len(bins)-1)]
plot_df['feature_value_bin'] = pd.cut(plot_df['feature_value_pct'], bins=bins, labels=labels, include_lowest=True)

# --- NEW: discrete RdBu colors sampled across the scale (low=blue → high=red) ---
colors = px.colors.sample_colorscale('RdBu_r', np.linspace(0, 1, len(labels)))

# Create beeswarm-style scatter plot (unchanged except color args)
fig = px.strip(
    plot_df,
    x='shap_value',
    y='feature',
    color='feature_value_bin',                 # NEW: use binned category
    title='🎯 SHAP Impact: Feature Value vs SHAP Value',
    labels={
        'shap_value': 'SHAP Value (impact on model output)',
        'feature': 'Feature',
        'feature_value_bin': 'Feature value (within-feature percentile)'
    },
    color_discrete_sequence=colors,            # NEW: discrete RdBu palette
    category_orders={'feature_value_bin': labels},  # keep legend in low→high order
    height=700
)

fig.update_traces(marker=dict(size=4, opacity=0.6))
fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="No impact")
fig.show()

print("\n💡 Interpretation:")
print("   - Red = higher values for that feature (relative to its own distribution)")
print("   - Blue = lower values for that feature")
print("   - Distance from 0 = Magnitude of impact")

In [ ]:
# SHAP dependence plot for top feature (interactive)
top_feature = shap_importance.iloc[0]['feature']
feat_idx = all_features.index(top_feature)

dependence_df = pd.DataFrame({
    'feature_value': X_shap[top_feature].values,
    'shap_value': shap_values[:, feat_idx],
    'target': y_test.loc[X_shap.index].values
})

fig = px.scatter(
    dependence_df,
    x='feature_value',
    y='shap_value',
    color='target',
    title=f'🎯 SHAP Dependence Plot: {top_feature}',
    labels={
        'feature_value': f'{top_feature} (Feature Value)',
        'shap_value': 'SHAP Value (Impact on Prediction)',
        'target': 'Actual Position'
    },
    color_continuous_scale='Viridis',
    opacity=0.6,
    height=500
)

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

print(f"\n💡 This shows how {top_feature} impacts predictions")
print("   Points above 0 = Feature increases predicted position (worse)")
print("   Points below 0 = Feature decreases predicted position (better)")

---
## 6️⃣ Consolidated Feature Ranking

In [ ]:
# Combine all importance measures
# Normalize to 0-1 scale for comparison
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

consolidated = pd.DataFrame({'feature': all_features})

# Add normalized scores
consolidated = consolidated.merge(
    f_importance[['feature', 'f_score']].assign(f_score_norm=lambda x: normalize(x['f_score'])),
    on='feature'
)

consolidated = consolidated.merge(
    mi_importance[['feature', 'mi_score']].assign(mi_score_norm=lambda x: normalize(x['mi_score'])),
    on='feature'
)

consolidated = consolidated.merge(
    rf_importance[['feature', 'importance']].rename(columns={'importance': 'rf_importance'})
    .assign(rf_importance_norm=lambda x: normalize(x['rf_importance'])),
    on='feature'
)

consolidated = consolidated.merge(
    perm_df[['feature', 'importance_mean']].rename(columns={'importance_mean': 'perm_importance'})
    .assign(perm_importance_norm=lambda x: normalize(x['perm_importance'])),
    on='feature'
)

# Compute average rank
consolidated['avg_importance'] = consolidated[[
    'f_score_norm', 'mi_score_norm', 'rf_importance_norm', 'perm_importance_norm'
]].mean(axis=1)

consolidated = consolidated.sort_values('avg_importance', ascending=False)

print("\n🏆 CONSOLIDATED FEATURE RANKING (Top 15):")
print(consolidated[['feature', 'avg_importance']].head(15).to_string(index=False))

In [ ]:
# Plot consolidated ranking
top_consolidated = consolidated.head(15)

fig = px.bar(
    top_consolidated,
    x='avg_importance',
    y='feature',
    orientation='h',
    title='🏆 Consolidated Feature Importance (Average of 4 Methods)',
    labels={'avg_importance': 'Average Normalized Importance', 'feature': 'Feature'},
    color='avg_importance',
    color_continuous_scale='Reds',
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## 7️⃣ Feature Group Comparison

In [ ]:
# Assign group labels
def get_feature_group(feature):
    if feature in historical_features:
        return 'Historical'
    elif feature in telemetry_features:
        return 'Telemetry'
    elif feature in weather_features:
        return 'Weather'
    elif feature in tire_features:
        return 'Tire'
    else:
        return 'Other'

consolidated['group'] = consolidated['feature'].apply(get_feature_group)

# Average importance by group
group_importance = consolidated.groupby('group')['avg_importance'].mean().sort_values(ascending=False)

fig = px.bar(
    x=group_importance.values,
    y=group_importance.index,
    orientation='h',
    title='📊 Average Importance by Feature Group',
    labels={'x': 'Average Importance', 'y': 'Feature Group'},
    color=group_importance.values,
    color_continuous_scale='Blues',
    height=400
)
fig.show()

print("\n📊 Feature Group Rankings:")
for group, importance in group_importance.items():
    print(f"   {group:15s}: {importance:.3f}")